# Q-Mini Hindi 512 — Automatic Training Program

यह notebook आपके पहले से बने हुए **705,742-parameter, 512-token** initial model को ही आगे train करता है।

**आपको सिर्फ एक बार Colab में GPU चुनना है:** Runtime → Change runtime type → Hardware accelerator → GPU (T4 या उपलब्ध GPU) → Save. फिर नीचे का एकमात्र code cell Run करें।

यह cell Google Drive mount करेगा, Drive में आपका initial model खोजेगा, Wikimedia Hindi Wikipedia corpus के pinned files अपने-आप डाउनलोड और SHA-256 से verify करेगा, tokenizer तथा train/validation streams बनाएगा, और **20,000 optimizer steps** तक QAT pretraining शुरू करेगा। हर 10 steps पर verified checkpoint Drive में जाएगा; अगली बार cell चलाने पर वहीं से resume होगा।

Data source: Wikimedia Wikipedia Hindi subset 20231101.hi, declared licenses CC BY-SA 3.0 and GFDL. Corpus source/attribution details are stored in the Drive package.

**ईमानदार सीमा:** 0.7M parameters और 512 context वाला model दुनिया के बड़े LLMs के बराबर नहीं बन सकता। यह पहला Hindi text-pretraining चरण है, finished assistant नहीं। सवाल-जवाब में निर्देश मानने के लिए बाद में अलग licensed instruction/conversation dataset पर fine-tuning चाहिए।

In [ ]:
# Q-MINI HINDI 512 — ONE-CELL AUTO PRETRAINER
# Uses the existing verified initial model. Does not rebuild random weights.
# Downloads a pinned, license-declared Hindi Wikipedia corpus; trains and saves
# resumable checkpoints and the final Q2-v2 model to Google Drive.

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid

# -------------------------- Fixed, verified settings --------------------------
REPOSITORY_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
EXPECTED_SOURCE_REVISION = "60a76100515d505484020162efea212458810b2c"
MODEL_CONFIG_REL = "configs/qmini_light_0.7m.json"

DATASET_ID = "wikimedia/wikipedia"
DATASET_REVISION = "cbc964b7871470596b80f7db5602de3d96dc4708"
DATASET_SUBSET = "20231101.hi"
DATASET_PAGE = (
    "https://huggingface.co/datasets/wikimedia/wikipedia/tree/"
    + DATASET_REVISION + "/" + DATASET_SUBSET
)
DATASET_FILES = [
    {
        "name": "train-00000-of-00002.parquet",
        "url": (
            "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/"
            + DATASET_REVISION + "/" + DATASET_SUBSET
            + "/train-00000-of-00002.parquet"
        ),
        "sha256": "48001d229cb28d1e74c1f6bac76a1d0ccfc2ffa05c8fb9c2201a5b56108e7c58",
    },
    {
        "name": "train-00001-of-00002.parquet",
        "url": (
            "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/"
            + DATASET_REVISION + "/" + DATASET_SUBSET
            + "/train-00001-of-00002.parquet"
        ),
        "sha256": "6ea66949ce2be6be06059370816f778d97f31b263f8b235d805a445dc0584d20",
    },
]

VOCAB_SIZE = 4096
VALIDATION_FRACTION = 0.02
TRAIN_SEQUENCE_LENGTH = 512
TARGET_OPTIMIZER_STEPS = 20000
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
CHECKPOINT_EVERY = 10
EVALUATE_EVERY = 100
EVALUATION_BATCHES = 8
MIN_FREE_GPU_GIB = 8.0
LEASE_STALE_SECONDS = 180
LEASE_HEARTBEAT_SECONDS = 45

def run_checked(command, *, cwd=None, capture_output=False, env=None):
    command = [str(item) for item in command]
    print("$", " ".join(command))
    return subprocess.run(
        command, cwd=str(cwd) if cwd else None, check=True,
        text=True, capture_output=capture_output, env=env,
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )

def download_verified(record, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected = record["sha256"]
    if destination.is_file() and sha256_file(destination) == expected:
        print("Already downloaded and verified:", destination.name)
        return
    if destination.exists():
        destination.unlink()
    partial = destination.with_name(destination.name + ".part")
    partial.unlink(missing_ok=True)
    print("Downloading:", record["name"])
    request = Request(
        record["url"],
        headers={"User-Agent": "Q-Mini-Research-Training/1.0"},
    )
    total = 0
    with urlopen(request, timeout=90) as response, partial.open("wb") as output:
        while True:
            chunk = response.read(8 * 1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
            total += len(chunk)
            if total and total % (32 * 1024 * 1024) < len(chunk):
                print("  received", f"{total / (1024**2):.1f} MiB")
    actual = sha256_file(partial)
    if actual != expected:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f"Dataset SHA-256 mismatch for {record['name']}; refusing to train on an unverified file."
        )
    partial.replace(destination)
    print("Verified:", destination.name, f"({total / (1024**2):.1f} MiB)")

print("\n[1/8] Checking Drive and GPU...")
MY_DRIVE = Path("/content/drive/MyDrive")
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive is not connected. Allow the Drive permission prompt, then rerun this same cell.")

import torch
from packaging.version import Version

if Version(torch.__version__.split("+", 1)[0]) < Version("2.10.0"):
    raise RuntimeError(f"Q-Mini needs PyTorch >=2.10.0; this runtime has {torch.__version__}.")
if not torch.cuda.is_available():
    raise RuntimeError(
        "TRAINING NOT STARTED: this Colab session has CPU-only PyTorch. "
        "Choose Runtime > Change runtime type > Hardware accelerator > GPU, "
        "save, then run this same cell. The existing model in Drive is safe."
    )

gpu_name = torch.cuda.get_device_name(0)
free_gpu_bytes, total_gpu_bytes = torch.cuda.mem_get_info(0)
print("PyTorch:", torch.__version__)
print("GPU:", gpu_name)
print(f"GPU memory: {free_gpu_bytes / (1024**3):.2f} GiB free / {total_gpu_bytes / (1024**3):.2f} GiB total")
if free_gpu_bytes < MIN_FREE_GPU_GIB * (1024**3):
    raise RuntimeError(
        f"TRAINING NOT STARTED: less than {MIN_FREE_GPU_GIB:.1f} GiB of GPU memory is free. "
        "Use a GPU runtime with more free memory. Do not lower this safety check."
    )

# Preserve the Colab-provided PyTorch installation.
run_checked([
    sys.executable, "-m", "pip", "install", "--quiet",
    "numpy>=1.24", "tokenizers>=0.15", "pyarrow>=15.0", "packaging>=23",
])

print("\n[2/8] Finding the existing initial model in Google Drive...")
INITIAL_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "initial_brains"
if not INITIAL_ROOT.is_dir():
    raise FileNotFoundError(f"Initial model folder is missing: {INITIAL_ROOT}")

valid_initials = []
for folder in INITIAL_ROOT.glob("QMini-qmini_light_0.7m-Q2v2-initial-*"):
    manifest_path = folder / "model_manifest.json"
    config_path = folder / "config.json"
    model_path = folder / "model.qmini.npz"
    if not (manifest_path.is_file() and config_path.is_file() and model_path.is_file()):
        continue
    try:
        candidate_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        candidate_config = json.loads(config_path.read_text(encoding="utf-8"))
        if (
            candidate_manifest.get("format") == "qmini-initial-brain-v1"
            and candidate_manifest.get("status") == "untrained_initialization"
            and candidate_manifest.get("training_steps") == 0
            and candidate_manifest.get("optimizer_steps") == 0
            and candidate_manifest.get("parameters") == 705742
            and candidate_config.get("max_context") == 512
            and candidate_config.get("vocab_size") == VOCAB_SIZE
            and candidate_manifest.get("packed_export_sha256") == sha256_file(model_path)
        ):
            valid_initials.append((folder.stat().st_mtime, folder, candidate_manifest))
    except (OSError, ValueError, TypeError):
        continue

if not valid_initials:
    raise FileNotFoundError(
        "No verified 705,742-parameter, 512-context initial model was found in "
        "MyDrive/QMini-Light-0.7M/initial_brains. The training program will not create "
        "a substitute random model."
    )
valid_initials.sort(key=lambda item: item[0], reverse=True)
_initial_time, INITIAL_DIR_SOURCE, INITIAL_MANIFEST = valid_initials[0]
INITIAL_MODEL_SOURCE = INITIAL_DIR_SOURCE / "model.qmini.npz"
INITIAL_SHA256 = INITIAL_MANIFEST["packed_export_sha256"]
SOURCE_REVISION = INITIAL_MANIFEST.get("source_revision")
if SOURCE_REVISION != EXPECTED_SOURCE_REVISION:
    raise RuntimeError(
        "The initial model's source revision differs from the reviewed training source. "
        f"Model revision={SOURCE_REVISION!r}; expected={EXPECTED_SOURCE_REVISION!r}. "
        "Refusing to mix model weights and code from different revisions."
    )
print("Using existing initial model:", INITIAL_DIR_SOURCE.name)
print("Initial model SHA-256:", INITIAL_SHA256)
print("Parameters: 705,742 | context: 512 | training steps: 0")

print("\n[3/8] Fetching the exact Q-Mini source revision...")
REPO_DIR = Path("/content") / ("QMini-training-source-" + uuid.uuid4().hex[:8])
REPO_DIR.mkdir(parents=True, exist_ok=False)
run_checked(["git", "init"], cwd=REPO_DIR)
run_checked(["git", "remote", "add", "origin", REPOSITORY_URL], cwd=REPO_DIR)
run_checked(["git", "fetch", "--depth", "1", "origin", SOURCE_REVISION], cwd=REPO_DIR)
run_checked(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR)
actual_revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("Downloaded Q-Mini source revision does not match the initial model manifest.")
if not (REPO_DIR / "qmini" / "cli.py").is_file():
    raise FileNotFoundError("The pinned Q-Mini source checkout is incomplete.")
sys.path.insert(0, str(REPO_DIR))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")

from qmini.config import QMiniConfig
from qmini.export import verify_packed_export
from qmini.tokenizer import load_tokenizer
from qmini.text import SPECIAL_TOKENS

base_config = QMiniConfig.from_json(REPO_DIR / MODEL_CONFIG_REL)
if base_config.max_context != TRAIN_SEQUENCE_LENGTH:
    raise RuntimeError("The GitHub model config does not have the required 512-token context.")
if base_config.estimated_parameter_count() != INITIAL_MANIFEST["parameters"]:
    raise RuntimeError("GitHub model config and saved initial model parameter count differ.")
initial_report = verify_packed_export(INITIAL_MODEL_SOURCE)
if (
    initial_report.get("format") != "qmini-packed-v2"
    or initial_report.get("q2_version") != 2
    or initial_report.get("parameter_count") != base_config.estimated_parameter_count()
):
    raise RuntimeError("The existing model failed Q2-v2 verification.")

print("\n[4/8] Downloading and verifying the Hindi Wikipedia corpus...")
DATA_DIR = MY_DRIVE / "QMini-Light-0.7M" / "datasets" / "wikimedia_20231101_hi"
DATA_DIR.mkdir(parents=True, exist_ok=True)
for record in DATASET_FILES:
    download_verified(record, DATA_DIR / record["name"])

SOURCE_METADATA_PATH = Path("/content/qmini-hindi-source-provenance.json")
source_metadata = {
    "format": "qmini-source-provenance-v1",
    "sources": [
        {
            "path": record["name"],
            "source_name": "Wikimedia Wikipedia Hindi subset 20231101.hi",
            "source_uri": DATASET_PAGE,
            "license_spdx": "CC-BY-SA-3.0 AND GFDL",
            "collected_at": datetime.now(timezone.utc).date().isoformat(),
            "language": "hi",
            "transformation_version": "qmini-parquet-source-v1",
        }
        for record in DATASET_FILES
    ],
}
write_json(SOURCE_METADATA_PATH, source_metadata)

data_fingerprints = [
    {"name": item["name"], "sha256": sha256_file(DATA_DIR / item["name"])}
    for item in DATASET_FILES
]
package_identity = {
    "format": "qmini-hindi-training-package-v1",
    "source_revision": SOURCE_REVISION,
    "initial_model_sha256": INITIAL_SHA256,
    "dataset": DATASET_PAGE,
    "dataset_files": data_fingerprints,
    "vocab_size": VOCAB_SIZE,
    "context_length": TRAIN_SEQUENCE_LENGTH,
    "validation_fraction": VALIDATION_FRACTION,
    "target_optimizer_steps": TARGET_OPTIMIZER_STEPS,
}
PACKAGE_KEY = hashlib.sha256(
    json.dumps(package_identity, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
PACKAGE_NAME = "QMini-Hindi-512-Q2v2-" + PACKAGE_KEY
PACKAGE_ROOT = MY_DRIVE / "QMini-Light-0.7M" / "training_ready"
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
PACKAGE_DIR = PACKAGE_ROOT / PACKAGE_NAME

print("Corpus files:", len(data_fingerprints))
print("Corpus size:", f"{sum((DATA_DIR / r['name']).stat().st_size for r in DATASET_FILES) / (1024**2):.1f} MiB")
print("Training package:", PACKAGE_DIR)

def verify_training_package(package_dir):
    package_dir = Path(package_dir)
    initial_dir = package_dir / "initial_brain"
    tokenizer_path = package_dir / "tokenizer" / "tokenizer.json"
    dataset_dir = package_dir / "dataset"
    runtime_dir = package_dir / "training_runtime"
    runtime_manifest_path = runtime_dir / "TRAINING_RUNTIME.json"
    config_path = initial_dir / "config.json"
    model_path = initial_dir / "model.qmini.npz"
    manifest_path = initial_dir / "model_manifest.json"
    progress_path = package_dir / "training_state" / "training_progress.json"
    required = [
        config_path, model_path, manifest_path, tokenizer_path,
        Path(str(tokenizer_path) + ".qmini.json"),
        package_dir / "active_config.json",
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        dataset_dir / "train.mask.bin", dataset_dir / "valid.mask.bin",
        dataset_dir / "dataset.json",
        runtime_manifest_path, runtime_dir / "qmini" / "train.py",
        runtime_dir / "qmini" / "checkpointing.py",
        progress_path,
    ]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Training package is incomplete: " + ", ".join(missing))

    config = QMiniConfig.from_json(config_path)
    active_config = QMiniConfig.from_json(package_dir / "active_config.json")
    if config.to_dict() != active_config.to_dict():
        raise RuntimeError("Tokenizer changed the model architecture or vocabulary unexpectedly.")
    if config.max_context != 512 or config.vocab_size != VOCAB_SIZE:
        raise RuntimeError("Training package does not preserve 512 context and 4096 vocabulary.")
    if config.estimated_parameter_count() != 705742:
        raise RuntimeError("Training package parameter count changed unexpectedly.")
    loaded_tokenizer = load_tokenizer(tokenizer_path)
    if loaded_tokenizer.get_vocab_size() != VOCAB_SIZE:
        raise RuntimeError("Tokenizer vocabulary does not match the initial model.")
    for token in SPECIAL_TOKENS:
        if loaded_tokenizer.token_to_id(token) is None:
            raise RuntimeError(f"Required structural token is missing: {token}")

    _verify_initial_brain_artifact(model_path, config)
    _verify_prepared_dataset_manifest(
        dataset_dir / "train.bin", dataset_dir / "valid.bin",
        tokenizer_path, dataset_dir / "train.mask.bin",
        dataset_dir / "valid.mask.bin", config.vocab_size,
    )
    train_array = _load_tokens(dataset_dir / "train.bin", "Training")
    valid_array = _load_tokens(dataset_dir / "valid.bin", "Validation")
    _validate_token_ids(train_array, config.vocab_size, "Training")
    _validate_token_ids(valid_array, config.vocab_size, "Validation")
    if len(train_array) < 514 or len(valid_array) < 514:
        raise RuntimeError("Prepared data is too small for 512-token training sequences.")
    if (dataset_dir / "train.mask.bin").stat().st_size != (dataset_dir / "train.bin").stat().st_size // 2:
        raise RuntimeError("Training mask size does not match the token stream.")
    if (dataset_dir / "valid.mask.bin").stat().st_size != (dataset_dir / "valid.bin").stat().st_size // 2:
        raise RuntimeError("Validation mask size does not match the token stream.")

    runtime_manifest = json.loads(runtime_manifest_path.read_text(encoding="utf-8"))
    if runtime_manifest.get("enabled") is not False or runtime_manifest.get("requires_manual_activation") is not True:
        raise RuntimeError("Training runtime was unexpectedly enabled in the package.")
    for record in runtime_manifest.get("source_files", []):
        relative = Path(record["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Unsafe path in training runtime manifest.")
        source = runtime_dir / relative
        if not source.is_file() or sha256_file(source) != record["sha256"]:
            raise RuntimeError(f"Training source hash mismatch: {relative}")
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if model_manifest.get("training_runtime_manifest_sha256") != sha256_file(runtime_manifest_path):
        raise RuntimeError("Training runtime manifest hash does not match model manifest.")
    if model_manifest.get("packed_export_sha256") != sha256_file(model_path):
        raise RuntimeError("Initial model fingerprint changed inside the training package.")
    return config, model_manifest, train_array, valid_array

if PACKAGE_DIR.is_dir():
    print("\nExisting training package found; verifying and reusing it.")
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
else:
    print("\n[5/8] Creating tokenizer, train/validation streams and a resumable package...")
    LOCAL_PACKAGE = Path("/content") / (PACKAGE_NAME + "-staging-" + uuid.uuid4().hex[:6])
    LOCAL_PACKAGE.mkdir(parents=True, exist_ok=False)
    (LOCAL_PACKAGE / "tokenizer").mkdir()
    (LOCAL_PACKAGE / "dataset").mkdir()
    (LOCAL_PACKAGE / "initial_brain").mkdir()
    (LOCAL_PACKAGE / "training_state").mkdir()

    TOKENIZER_PATH = LOCAL_PACKAGE / "tokenizer" / "tokenizer.json"
    ACTIVE_CONFIG_PATH = LOCAL_PACKAGE / "active_config.json"
    DATASET_OUT_DIR = LOCAL_PACKAGE / "dataset"

    run_checked([
        sys.executable, "-m", "qmini.cli", "tokenizer",
        "--data-dir", DATA_DIR,
        "--output", TOKENIZER_PATH,
        "--vocab-size", str(VOCAB_SIZE),
        "--base-config", REPO_DIR / MODEL_CONFIG_REL,
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", "0",
        "--source-metadata", SOURCE_METADATA_PATH,
        "--config-output", ACTIVE_CONFIG_PATH,
    ], cwd=REPO_DIR)

    active_config = QMiniConfig.from_json(ACTIVE_CONFIG_PATH)
    if active_config.to_dict() != base_config.to_dict():
        raise RuntimeError(
            "Tokenizer vocabulary/config differs from the saved initial model. "
            "No weights will be changed; inspect the corpus/tokenizer before continuing."
        )

    run_checked([
        sys.executable, "-m", "qmini.cli", "prepare",
        "--data-dir", DATA_DIR,
        "--tokenizer", TOKENIZER_PATH,
        "--output-dir", DATASET_OUT_DIR,
        "--valid-fraction", str(VALIDATION_FRACTION),
        "--near-duplicate-hamming-distance", "0",
        "--loss-mask", "all",
        "--source-metadata", SOURCE_METADATA_PATH,
    ], cwd=REPO_DIR)

    # Reuse the exact initial artifact from the previous successful build.
    for filename in ("config.json", "model.qmini.npz", "model_manifest.json"):
        shutil.copy2(INITIAL_DIR_SOURCE / filename, LOCAL_PACKAGE / "initial_brain" / filename)

    # Keep the complete source checkout and a hash-checked training-runtime bundle.
    shutil.copytree(
        REPO_DIR, LOCAL_PACKAGE / "project_source",
        ignore=shutil.ignore_patterns(
            ".git", "__pycache__", "*.pyc", ".pytest_cache",
            ".venv", "venv", ".mypy_cache", ".ruff_cache",
        ),
    )
    RUNTIME_DIR_LOCAL = LOCAL_PACKAGE / "training_runtime"
    RUNTIME_DIR_LOCAL.mkdir()
    shutil.copytree(
        REPO_DIR / "qmini", RUNTIME_DIR_LOCAL / "qmini",
        ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "*.so", "*.pyd"),
    )
    shutil.copy2(REPO_DIR / "pyproject.toml", RUNTIME_DIR_LOCAL / "pyproject.toml")
    write_json(RUNTIME_DIR_LOCAL / "TRAINING_DEFAULTS.json", {
        "format": "qmini-training-defaults-v1",
        "enabled": False,
        "sequence_length": TRAIN_SEQUENCE_LENGTH,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "checkpoint_every_optimizer_steps": CHECKPOINT_EVERY,
        "evaluation_every_optimizer_steps": EVALUATE_EVERY,
        "total_optimizer_steps": TARGET_OPTIMIZER_STEPS,
        "minimum_free_gpu_gib": MIN_FREE_GPU_GIB,
    })
    (RUNTIME_DIR_LOCAL / "README.md").write_text(
        "# Q-Mini Hindi training runtime\n\n"
        "Exact qmini source matching the step-zero initial model source revision. "
        "Training settings are recorded in TRAINING_DEFAULTS.json. Training is "
        "activated only by the one-click training notebook.\n",
        encoding="utf-8",
    )

    runtime_files = []
    for source in sorted(p for p in RUNTIME_DIR_LOCAL.rglob("*") if p.is_file() and p.name != "TRAINING_RUNTIME.json"):
        runtime_files.append({
            "path": source.relative_to(RUNTIME_DIR_LOCAL).as_posix(),
            "sha256": sha256_file(source),
            "size_bytes": source.stat().st_size,
        })
    runtime_manifest = {
        "format": "qmini-training-runtime-v1",
        "enabled": False,
        "requires_manual_activation": True,
        "entrypoint": "qmini.train.train_model",
        "checkpoint_backend": "qmini.checkpointing",
        "source_revision": SOURCE_REVISION,
        "source_files": runtime_files,
    }
    write_json(RUNTIME_DIR_LOCAL / "TRAINING_RUNTIME.json", runtime_manifest)

    write_json(LOCAL_PACKAGE / "training_state" / "training_progress.json", {
        "format": "qmini-training-progress-v1",
        "status": "not_started",
        "current_step": 0,
        "target_steps": TARGET_OPTIMIZER_STEPS,
        "progress_percent": 0.0,
        "tokens_seen": 0,
        "updated_at": datetime.now(timezone.utc).isoformat(),
    })

    # Bind the existing initial weights to this exact tokenizer/data/runtime.
    manifest_path = LOCAL_PACKAGE / "initial_brain" / "model_manifest.json"
    model_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    dataset_manifest_path = DATASET_OUT_DIR / "dataset.json"
    dataset_manifest = json.loads(dataset_manifest_path.read_text(encoding="utf-8"))
    tokenizer_sidecar = Path(str(TOKENIZER_PATH) + ".qmini.json")
    model_manifest.update({
        "tokenizer_included": True,
        "tokenizer_relative_path": "../tokenizer/tokenizer.json",
        "tokenizer_sha256": sha256_file(TOKENIZER_PATH),
        "tokenizer_manifest_sha256": sha256_file(tokenizer_sidecar),
        "prepared_dataset_relative_path": "../dataset",
        "prepared_dataset_manifest_sha256": sha256_file(dataset_manifest_path),
        "train_tokens_sha256": dataset_manifest["train_tokens_sha256"],
        "valid_tokens_sha256": dataset_manifest["valid_tokens_sha256"],
        "train_mask_sha256": dataset_manifest.get("train_mask_sha256"),
        "valid_mask_sha256": dataset_manifest.get("valid_mask_sha256"),
        "corpus_sha256": dataset_manifest.get("corpus_sha256"),
        "source_provenance_sha256": dataset_manifest.get("source_provenance_sha256"),
        "release_mode": bool(dataset_manifest.get("release_mode", False)),
        "training_runtime_included": True,
        "training_runtime_enabled": False,
        "training_runtime_manifest_sha256": sha256_file(RUNTIME_DIR_LOCAL / "TRAINING_RUNTIME.json"),
    })
    write_json(manifest_path, model_manifest)

    license_note = (
        "# Hindi Wikipedia training data source and attribution\n\n"
        + "- Dataset: " + DATASET_ID + " subset " + DATASET_SUBSET + "\n"
        + "- Pinned dataset revision: " + DATASET_REVISION + "\n"
        + "- Dataset page: " + DATASET_PAGE + "\n"
        + "- Source: Wikimedia Wikipedia contributors, Hindi Wikipedia.\n"
        + "- Declared dataset licenses on the source page: CC BY-SA 3.0 and GFDL.\n"
        + "- Both downloaded Parquet shard hashes are recorded in BUILD_REPORT.json.\n"
        + "- Before redistributing the corpus or a model trained on it, review and comply "
          "with the applicable attribution and share-alike terms. This note is not legal advice.\n"
    )
    (LOCAL_PACKAGE / "DATA_SOURCE_AND_LICENSE.md").write_text(license_note, encoding="utf-8")

    write_json(LOCAL_PACKAGE / "BUILD_REPORT.json", {
        "format": "qmini-hindi-training-package-report-v1",
        "source_revision": SOURCE_REVISION,
        "initial_model_source": INITIAL_DIR_SOURCE.name,
        "initial_model_sha256": INITIAL_SHA256,
        "dataset_page": DATASET_PAGE,
        "dataset_revision": DATASET_REVISION,
        "dataset_files": data_fingerprints,
        "context_length": TRAIN_SEQUENCE_LENGTH,
        "parameters": 705742,
        "vocabulary_size": VOCAB_SIZE,
        "validation_fraction": VALIDATION_FRACTION,
        "training": {
            "target_optimizer_steps": TARGET_OPTIMIZER_STEPS,
            "batch_size": BATCH_SIZE,
            "gradient_accumulation": GRAD_ACCUM,
            "sequence_length": TRAIN_SEQUENCE_LENGTH,
            "started": False,
        },
        "license_notice": "CC BY-SA 3.0 and GFDL; see DATA_SOURCE_AND_LICENSE.md",
    })

    verify_training_package(LOCAL_PACKAGE)
    DRIVE_STAGING = PACKAGE_ROOT / ("." + PACKAGE_NAME + ".staging-" + uuid.uuid4().hex[:8])
    try:
        shutil.copytree(LOCAL_PACKAGE, DRIVE_STAGING)
        verify_training_package(DRIVE_STAGING)
        DRIVE_STAGING.rename(PACKAGE_DIR)
    except Exception:
        shutil.rmtree(DRIVE_STAGING, ignore_errors=True)
        raise
    PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
    print("Training package created and Drive copy verified.")

# Re-verify persistent package every time before creating an optimizer.
PACKAGE_CONFIG, PACKAGE_MODEL_MANIFEST, train_tokens, valid_tokens = verify_training_package(PACKAGE_DIR)
INITIAL_DIR = PACKAGE_DIR / "initial_brain"
INITIAL_MODEL_PATH = INITIAL_DIR / "model.qmini.npz"
CONFIG_PATH = INITIAL_DIR / "config.json"
TOKENIZER_PATH = PACKAGE_DIR / "tokenizer" / "tokenizer.json"
DATASET_DIR = PACKAGE_DIR / "dataset"
TRAIN_BIN = DATASET_DIR / "train.bin"
VALID_BIN = DATASET_DIR / "valid.bin"
TRAIN_MASK = DATASET_DIR / "train.mask.bin"
VALID_MASK = DATASET_DIR / "valid.mask.bin"
RUNTIME_DIR = PACKAGE_DIR / "training_runtime"
MIRROR_DIR = PACKAGE_DIR / "training_state"
PROGRESS_PATH = MIRROR_DIR / "training_progress.json"
LOCAL_RUN_DIR = Path("/content/qmini-hindi-512-runs") / PACKAGE_NAME

from qmini.train import (
    _load_tokens,
    _validate_token_ids,
    _verify_initial_brain_artifact,
    _verify_prepared_dataset_manifest,
    train_model,
)
from qmini.checkpointing import (
    TrainingSessionLease,
    mark_training_status,
    restore_latest_checkpoint,
)
from qmini.export import verify_packed_export

print("\n[6/8] Checking package, disk space and checkpoint safety...")
assert PACKAGE_CONFIG.max_context == 512
assert PACKAGE_CONFIG.estimated_parameter_count() == 705742
_verify_initial_brain_artifact(INITIAL_MODEL_PATH, PACKAGE_CONFIG)
_verify_prepared_dataset_manifest(
    TRAIN_BIN, VALID_BIN, TOKENIZER_PATH, TRAIN_MASK, VALID_MASK, PACKAGE_CONFIG.vocab_size
)
checkpoint_estimate_bytes = int(PACKAGE_CONFIG.estimated_parameter_count() * 12 * 1.2 + 512 * 1024**2)
drive_required_bytes = 3 * checkpoint_estimate_bytes + 1024**3
local_required_bytes = 2 * checkpoint_estimate_bytes + 1024**3
drive_free_bytes = shutil.disk_usage(MY_DRIVE).free
local_free_bytes = shutil.disk_usage("/content").free
print(f"Drive free: {drive_free_bytes / (1024**3):.2f} GiB")
print(f"Colab local free: {local_free_bytes / (1024**3):.2f} GiB")
if drive_free_bytes < drive_required_bytes:
    raise RuntimeError("Not enough Google Drive space for verified checkpoint generations and model export.")
if local_free_bytes < local_required_bytes:
    raise RuntimeError("Not enough Colab temporary disk space for safe checkpoint creation.")

progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
print("Current saved training status:", progress_now.get("status"))
print("Last verified step:", progress_now.get("current_step", 0), "/", progress_now.get("target_steps"))
print("Train tokens:", f"{len(train_tokens):,}")
print("Validation tokens:", f"{len(valid_tokens):,}")
print("Training device:", gpu_name)
print("Sequence length:", TRAIN_SEQUENCE_LENGTH)
print("Target optimizer steps:", TARGET_OPTIMIZER_STEPS)

print("\n[7/8] Training/resuming the existing initial model...")
progress_now = json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
if progress_now.get("status") == "completed" and progress_now.get("final_model_sha256"):
    print("This exact dataset/model package is already fully trained. No duplicate training run will be started.")
    final_drive_dir = None
else:
    LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
    session_started = datetime.now(timezone.utc).isoformat()
    with TrainingSessionLease(
        MIRROR_DIR,
        stale_after_seconds=LEASE_STALE_SECONDS,
        heartbeat_seconds=LEASE_HEARTBEAT_SECONDS,
    ) as lease:
        previous_progress = (
            json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
            if PROGRESS_PATH.is_file() else {}
        )
        if previous_progress.get("status") == "running":
            mark_training_status(
                MIRROR_DIR, "interrupted",
                note="Previous Colab runtime ended without releasing its training lease.",
            )

        restored = restore_latest_checkpoint(MIRROR_DIR, LOCAL_RUN_DIR)
        if restored is None:
            current_progress = (
                json.loads(PROGRESS_PATH.read_text(encoding="utf-8"))
                if PROGRESS_PATH.is_file() else {}
            )
            if current_progress.get("current_step", 0) not in (0, None):
                raise RuntimeError(
                    "Drive progress shows training steps, but no verified checkpoint can be restored. "
                    "Refusing to restart from random weights."
                )
            resume_run = False
            extend_schedule = False
            current_step = 0
        else:
            resume_run = True
            extend_schedule = TARGET_OPTIMIZER_STEPS > restored["target_steps"]
            current_step = restored["step"]
            if current_step > TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("Saved optimizer step is above this notebook's target.")
            if restored["target_steps"] > TARGET_OPTIMIZER_STEPS and current_step < TARGET_OPTIMIZER_STEPS:
                raise RuntimeError("The new target is below the previous run's target; refusing to shorten the schedule.")

        mark_training_status(
            MIRROR_DIR, "running",
            note=f"Q-Mini Hindi 512 training session started at {session_started}.",
        )
        print("Resume:", resume_run)
        print("Starting step:", current_step, "/", TARGET_OPTIMIZER_STEPS)

        FINAL_LOCAL_DIR = train_model(
            config=PACKAGE_CONFIG,
            train_bin=TRAIN_BIN,
            valid_bin=VALID_BIN,
            output_dir=LOCAL_RUN_DIR,
            steps=TARGET_OPTIMIZER_STEPS,
            batch_size=BATCH_SIZE,
            seq_len=TRAIN_SEQUENCE_LENGTH,
            grad_accum=GRAD_ACCUM,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            eval_every=EVALUATE_EVERY,
            save_every=CHECKPOINT_EVERY,
            eval_batches=EVALUATION_BATCHES,
            resume=resume_run,
            tokenizer_path=TOKENIZER_PATH,
            extend_schedule=extend_schedule,
            train_mask_bin=TRAIN_MASK,
            valid_mask_bin=VALID_MASK,
            init_from_initial=None if resume_run else INITIAL_MODEL_PATH,
            checkpoint_mirror_dir=MIRROR_DIR,
            checkpoint_session_id=lease.session_id,
        )

        final_manifest_path = FINAL_LOCAL_DIR / "model_manifest.json"
        if not final_manifest_path.is_file():
            raise FileNotFoundError("Training did not produce a final_model manifest.")
        final_manifest = json.loads(final_manifest_path.read_text(encoding="utf-8"))
        final_packed_path = FINAL_LOCAL_DIR / final_manifest["packed_export"]
        final_report = verify_packed_export(final_packed_path)
        if (
            final_report.get("format") != "qmini-packed-v2"
            or final_report.get("q2_version") != 2
            or final_report.get("parameter_count") != PACKAGE_CONFIG.estimated_parameter_count()
        ):
            raise RuntimeError("Final trained Q2-v2 export failed format/parameter verification.")
        if sha256_file(final_packed_path) != final_manifest.get("packed_export_sha256"):
            raise RuntimeError("Final trained model SHA-256 differs from its manifest.")

        final_step = int(final_manifest["step"])
        trained_root = PACKAGE_DIR / "trained_models"
        trained_root.mkdir(parents=True, exist_ok=True)
        final_name = f"QMini-Hindi-512-trained-step-{final_step:012d}"
        final_drive_dir = trained_root / final_name
        staging_dir = trained_root / ("." + final_name + ".staging-" + uuid.uuid4().hex[:8])

        if final_drive_dir.exists():
            existing_manifest = json.loads((final_drive_dir / "model_manifest.json").read_text(encoding="utf-8"))
            existing_packed = final_drive_dir / existing_manifest["packed_export"]
            if (
                existing_manifest.get("step") != final_step
                or existing_manifest.get("packed_export_sha256") != final_manifest.get("packed_export_sha256")
                or sha256_file(existing_packed) != existing_manifest.get("packed_export_sha256")
            ):
                raise FileExistsError("An existing trained model conflicts with this final optimizer step.")
            verify_packed_export(existing_packed)
        else:
            try:
                shutil.copytree(FINAL_LOCAL_DIR, staging_dir)
                copied_manifest = json.loads((staging_dir / "model_manifest.json").read_text(encoding="utf-8"))
                copied_packed = staging_dir / copied_manifest["packed_export"]
                if sha256_file(copied_packed) != copied_manifest.get("packed_export_sha256"):
                    raise RuntimeError("Final model Drive-copy SHA-256 verification failed.")
                verify_packed_export(copied_packed)
                staging_dir.rename(final_drive_dir)
            except Exception:
                shutil.rmtree(staging_dir, ignore_errors=True)
                raise

        directory_digest = hashlib.sha256()
        for item in sorted(final_drive_dir.rglob("*")):
            if item.is_file():
                directory_digest.update(item.relative_to(final_drive_dir).as_posix().encode("utf-8"))
                directory_digest.update(sha256_file(item).encode("ascii"))
        final_model_digest = directory_digest.hexdigest()
        final_progress = mark_training_status(
            MIRROR_DIR, "completed",
            final_model_sha256=final_model_digest,
            note=f"Verified trained Q2-v2 model stored at trained_models/{final_name}",
        )

    print("\n[8/8] Q-MINI HINDI TRAINING COMPLETE")
    print("Optimizer step:", final_progress["current_step"], "/", final_progress["target_steps"])
    print("Tokens seen:", f"{final_progress['tokens_seen']:,}")
    print("Trained model:", final_drive_dir)
    print("Model file:", final_drive_dir / "model.qmini.npz")
    print("Verified checkpoints:", MIRROR_DIR)
    print("Progress file:", PROGRESS_PATH)
    print("\nTraining updates were made to the saved initial model. The original initial model remains unchanged.")
